In [1]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

In [5]:
import contextlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
from ipywidgets import widgets
from natsort import natsort_keygen

from latent_dynamics.dayang.activations import Activations, extract_activations
from latent_dynamics.dayang.data import DATASET_REGISTRY, load_dataset_from_spec
from latent_dynamics.dayang.model import MODEL_REGISTRY, load_model_and_tokenizer
from latent_dynamics.dayang.projections import compute_layerwise_pca, plot_layerwise_pca, plot_layerwise_pca_ratio
from latent_dynamics.dayang.scoring import compute_layerwise_score, plot_layerwise_score

torch.set_grad_enabled(False)

print(f"Available models:{'\n - '.join([''] + list(MODEL_REGISTRY))}")
print(f"Available datasets:{'\n - '.join([''] + list(DATASET_REGISTRY))}")

Available models:
 - gemma3_270m
 - qwen3_0.6b
 - qwen3_8b
 - llama3.1_8b
 - gemma3_4b
Available datasets:
 - wildjailbreak/train/vanilla
 - wildjailbreak/train/adversarial
 - wildjailbreak/eval/adversarial
 - beavertails330k/train
 - beavertails330k/test
 - beavertails30k/train
 - beavertails30k/test
 - sorry_bench
 - xstest


# Manual analysis

In [ ]:
dataset = load_dataset_from_spec("xstest", max_samples=200)

In [ ]:
model, tokenizer = load_model_and_tokenizer("gemma3_270m")

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Loaded model: google/gemma-3-270m-it
  Number of hidden layers:         18
  Size of hidden layers:           640
  Size of activations (per token): 47 KB
  Model dtype:                     torch.bfloat16
  Device:                          cpu


In [ ]:
activations = extract_activations(model, tokenizer, dataset)
activations.metadata

## Projection analysis

In [ ]:
pcas = compute_layerwise_pca(activations, pool_method="last")
plot_layerwise_pca_ratio(pcas)

In [ ]:
plot_layerwise_pca(activations, pcas, pool_method="last")

## Scoring analysis

In [ ]:
scorers = compute_layerwise_score(activations, method="difference_in_mean")

In [ ]:
plot_layerwise_score(activations, scorers, pool_method="last")

# Interactive analysis

## Projection analysis

In [ ]:
def analyze_layerwise_pca():
    path_activations = Path("../activations")
    state = {"activations": None, "pcas": {}}

    def get_activation_folders(path: Path):
        return sorted((str(p) for p in path.glob("**/*") if (p / "metadata.json").exists()), key=natsort_keygen())

    def _str_to_slice(s):
        try:
            return slice(*map(lambda x: int(x.strip()) if x.strip() else None, s.split(":")))
        except Exception as e:
            print(f"Invalid slice format: {s}. Expected format like '1:5' or '::2'. Error: {e}")
            return None

    # Layer 1: Extraction
    models = list(MODEL_REGISTRY.keys())
    datasets = list(DATASET_REGISTRY.keys())
    w_model = widgets.Dropdown(options=models, value=models[0], description="Model")
    w_dataset = widgets.Dropdown(options=datasets, value=datasets[0], description="Dataset")
    w_max_samples = widgets.IntText(value=200, min=1, description="Max samples:")
    w_include_response = widgets.Dropdown(
        options=[False, True, "Sorry", "Sure"], value=False, description="Include response"
    )
    w_apply_chat_template = widgets.Checkbox(value=False, description="Apply chat template")
    btn_extract = widgets.Button(description="Extract Activations", button_style="primary")
    col_extract = widgets.VBox(
        [
            w_model,
            w_dataset,
            w_max_samples,
            w_include_response,
            w_apply_chat_template,
            btn_extract,
        ]
    )

    w_path = widgets.Combobox(options=get_activation_folders(path_activations), description="Path")
    btn_save = widgets.Button(description="Save Activations", button_style="success", disabled=True)
    btn_load = widgets.Button(description="Load Activations", button_style="info")
    col_load = widgets.VBox(
        [
            w_path,
            widgets.HBox([btn_save, btn_load]),
        ]
    )

    out_extract = widgets.Output()
    layer1 = widgets.VBox([widgets.HBox([col_extract, col_load]), out_extract])

    # Layer 2: Analysis Setup
    w_pca_safe = widgets.SelectMultiple(description="Safe samples")
    w_pca_unsafe = widgets.SelectMultiple(description="Unsafe samples")
    w_pca_pool = widgets.Dropdown(
        options=["all", "first", "mid", "last", "mean", "slice"], value="last", description="Tokens"
    )
    w_pca_pool_slice = widgets.Text(value="::", description="Tokens slice")
    w_pca_exclude_bos = widgets.Checkbox(value=True, description="Exclude BOS token")
    w_pca_exclude_special = widgets.Checkbox(value=True, description="Exclude special tokens")
    col_pca = widgets.VBox(
        [
            widgets.HTML("<b>PCA</b>"),
            w_pca_safe,
            w_pca_unsafe,
            w_pca_pool,
            w_pca_pool_slice,
            w_pca_exclude_bos,
            w_pca_exclude_special,
        ]
    )

    w_vis_safe = widgets.SelectMultiple(description="Safe samples")
    w_vis_unsafe = widgets.SelectMultiple(description="Unsafe samples")
    w_vis_pool = widgets.Dropdown(
        options=["all", "first", "mid", "last", "mean", "slice"], value="last", description="Tokens"
    )
    w_vis_pool_slice = widgets.Text(value="::", description="Tokens slice")
    w_vis_exclude_bos = widgets.Checkbox(value=True, description="Exclude BOS token")
    w_vis_exclude_special = widgets.Checkbox(value=True, description="Exclude special tokens")
    col_vis = widgets.VBox(
        [
            widgets.HTML("<b>Visualization</b>"),
            w_vis_safe,
            w_vis_unsafe,
            w_vis_pool,
            w_vis_pool_slice,
            w_vis_exclude_bos,
            w_vis_exclude_special,
        ]
    )

    btn_plot = widgets.Button(description="Visualize", button_style="primary", disabled=True)
    out_plot = widgets.Output()
    layer2 = widgets.VBox([widgets.HBox([col_pca, col_vis]), btn_plot, out_plot])

    # Add interaction handlers for slice widget
    def update_slice_widgets(*args):
        w_pca_pool_slice.layout.display = None if w_pca_pool.value == "slice" else "none"
        w_vis_pool_slice.layout.display = None if w_vis_pool.value == "slice" else "none"

    w_pca_pool.observe(update_slice_widgets, names="value")
    w_vis_pool.observe(update_slice_widgets, names="value")
    update_slice_widgets()

    # Add interaction handlers for layer 1
    @contextlib.contextmanager
    def update_buttons():
        btn_extract.disabled = True
        btn_save.disabled = True
        btn_load.disabled = True
        btn_plot.disabled = True
        try:
            yield
        finally:
            btn_extract.disabled = False
            btn_save.disabled = state["activations"] is None
            btn_load.disabled = False
            btn_plot.disabled = state["activations"] is None

    def update_layer2_widgets(activations):
        samples_safe = activations.metadata[activations.metadata["is_safe"]].index.tolist()
        samples_unsafe = activations.metadata[~activations.metadata["is_safe"]].index.tolist()
        w_pca_safe.options = samples_safe
        w_pca_unsafe.options = samples_unsafe
        w_vis_safe.options = samples_safe
        w_vis_unsafe.options = samples_unsafe
        w_pca_safe.value = ()
        w_pca_unsafe.value = ()
        w_vis_safe.value = ()
        w_vis_unsafe.value = ()
        out_plot.clear_output()

    def do_extract(*args):
        # Run extraction
        with update_buttons(), out_extract:
            out_extract.clear_output(wait=True)
            model, tokenizer = load_model_and_tokenizer(w_model.value)
            dataset = load_dataset_from_spec(w_dataset.value, max_samples=w_max_samples.value)
            activations = extract_activations(
                model,
                tokenizer,
                dataset,
                include_response=w_include_response.value,
                apply_chat_template=w_apply_chat_template.value,
            )
            # Update state
            state["activations"] = activations
            state["pcas"] = {}
            # Update layer 2 widgets
            update_layer2_widgets(activations)

        state["is_processing"] = False
        update_buttons()

    def do_save(*args):
        path = w_path.value
        if not path or state["activations"] is None:
            return

        with update_buttons(), out_extract:
            out_extract.clear_output(wait=True)

            path = Path(path)
            if path.exists():
                print(f"Path '{path}' already exists. Please choose a different path.")
                return

            print(f"Saving activations to '{path}'...")
            state["activations"].save(path)
            print("Saved successfully!")
            # Update load options
            w_path.options = get_activation_folders(path)

    def do_load(*args):
        path = w_path.value
        if not path:
            return

        with update_buttons(), out_extract:
            out_extract.clear_output(wait=True)

            path = Path(path)
            if not path.exists():
                print(f"Path '{path}' does not exist. Please choose a valid path.")
                return

            print(f"Loading activations from '{path}'...")
            activations = Activations.load(path)
            print("Loaded successfully!")
            # Update state
            state["activations"] = activations
            state["pcas"] = {}
            # Update layer 2 widgets
            update_layer2_widgets(activations)

    btn_extract.on_click(do_extract)
    btn_save.on_click(do_save)
    btn_load.on_click(do_load)

    # Add interaction handlers for layer 2
    def do_plot(*args):
        if state["activations"] is None:
            return

        with out_plot:
            out_plot.clear_output(wait=True)

            # Select samples for PCA
            pca_sample_ids = list(w_pca_safe.value) + list(w_pca_unsafe.value)
            pca_activations = state["activations"].select(pca_sample_ids) if pca_sample_ids else state["activations"]
            # Determine pooling method
            pca_pool_method = _str_to_slice(w_pca_pool_slice.value) if w_pca_pool.value == "slice" else w_pca_pool.value
            if pca_pool_method is None:
                return
            # Compute and cache PCA
            cache_key = (tuple(sorted(pca_sample_ids)), pca_pool_method, w_pca_exclude_special.value)
            if cache_key not in state["pcas"]:
                state["pcas"][cache_key] = compute_layerwise_pca(
                    pca_activations,
                    pool_method=pca_pool_method,
                    exclude_bos=w_pca_exclude_bos.value,
                    exclude_special_tokens=w_pca_exclude_special.value,
                )
            # Plot PCA explained variance ratio
            plot_layerwise_pca_ratio(state["pcas"][cache_key])

            # Select samples for visualization
            vis_sample_ids = list(w_vis_safe.value) + list(w_vis_unsafe.value)
            vis_activations = state["activations"].select(vis_sample_ids) if vis_sample_ids else state["activations"]
            # Determine pooling method
            vis_pool_method = _str_to_slice(w_vis_pool_slice.value) if w_vis_pool.value == "slice" else w_vis_pool.value
            if vis_pool_method is None:
                return
            # Plot activations
            plot_layerwise_pca(
                vis_activations,
                pcas=state["pcas"][cache_key],
                pool_method=vis_pool_method,
                exclude_bos=w_vis_exclude_bos.value,
                exclude_special_tokens=w_vis_exclude_special.value,
            )

    btn_plot.on_click(do_plot)

    # Display widgets
    display(widgets.VBox([layer1, widgets.HTML("<hr>"), layer2]))


analyze_layerwise_pca()

In [ ]:
def analyze_layerwise_score():
    state = {"activations": None, "readers": {}}

    # Layer 1: Extraction
    models = list(MODEL_REGISTRY.keys())
    datasets = list(DATASET_REGISTRY.keys())

    w_model = widgets.Dropdown(options=models, value=models[0], description="Model")
    w_dataset = widgets.Dropdown(options=datasets, value=datasets[0], description="Dataset")
    w_max_samples = widgets.IntText(value=200, min=1, description="Max samples:")
    w_include_response = widgets.Dropdown(
        options=[False, True, "Sorry", "Sure"], value=False, description="Include response"
    )
    w_apply_chat_template = widgets.Checkbox(value=False, description="Apply chat template")
    btn_extract = widgets.Button(description="Extract Activations", button_style="primary")
    out_extract = widgets.Output()
    layer1 = widgets.VBox(
        [w_model, w_dataset, w_max_samples, w_include_response, w_apply_chat_template, btn_extract, out_extract]
    )

    # Layer 2: Analysis Setup
    w_reader_method = widgets.Dropdown(
        options=["difference_in_mean", "linear_probe"], value="difference_in_mean", description="Method"
    )
    w_pca_safe = widgets.SelectMultiple(description="Safe samples")
    w_pca_unsafe = widgets.SelectMultiple(description="Unsafe samples")
    w_pca_pool = widgets.Dropdown(options=["all", "first", "mid", "last", "mean"], value="last", description="Pool")
    w_pca_exclude_bos = widgets.Checkbox(value=True, description="Exclude BOS token")
    w_pca_exclude_special = widgets.Checkbox(value=True, description="Exclude special tokens")
    col_pca = widgets.VBox(
        [
            widgets.HTML("<b>PCA Setup</b>"),
            w_pca_safe,
            w_pca_unsafe,
            w_pca_pool,
            w_pca_exclude_bos,
            w_pca_exclude_special,
        ]
    )
    w_vis_safe = widgets.SelectMultiple(description="Safe samples")
    w_vis_unsafe = widgets.SelectMultiple(description="Unsafe samples")
    w_vis_pool = widgets.Dropdown(options=["all", "first", "mid", "last", "mean"], value="last", description="Pool")
    w_vis_exclude_bos = widgets.Checkbox(value=True, description="Exclude BOS token")
    w_vis_exclude_special = widgets.Checkbox(value=True, description="Exclude special tokens")
    col_vis = widgets.VBox(
        [
            widgets.HTML("<b>Visualization Setup</b>"),
            w_vis_safe,
            w_vis_unsafe,
            w_vis_pool,
            w_vis_exclude_bos,
            w_vis_exclude_special,
        ]
    )
    out_plot = widgets.Output()
    layer2 = widgets.VBox([w_reader_method, widgets.HBox([col_pca, col_vis]), out_plot])

    # Add interaction handlers for layer 1
    def on_extract(*args):
        btn_extract.disabled = True
        # Run extraction
        with out_extract:
            out_extract.clear_output(wait=True)
            model, tokenizer = load_model_and_tokenizer(w_model.value)
            dataset = load_dataset_from_spec(w_dataset.value, max_samples=w_max_samples.value)
            activations = extract_activations(
                model,
                tokenizer,
                dataset,
                include_response=w_include_response.value,
                apply_chat_template=w_apply_chat_template.value,
            )
        # Update state
        state["activations"] = activations
        state["readers"] = {}

        # Update layer 2 widgets
        samples_safe = activations.metadata[activations.metadata["is_safe"]].index.tolist()
        samples_unsafe = activations.metadata[~activations.metadata["is_safe"]].index.tolist()
        w_pca_safe.options = samples_safe
        w_pca_unsafe.options = samples_unsafe
        w_vis_safe.options = samples_safe
        w_vis_unsafe.options = samples_unsafe
        w_pca_safe.value = ()
        w_pca_unsafe.value = ()
        w_vis_safe.value = ()
        w_vis_unsafe.value = ()

        btn_extract.disabled = False

        update_plot()

    btn_extract.on_click(on_extract)

    # Add interaction handlers for layer 2
    def update_plot(*args):
        if state["activations"] is None:
            return

        with out_plot:
            out_plot.clear_output(wait=True)

            # Select samples for PCA
            pca_sample_ids = list(w_pca_safe.value) + list(w_pca_unsafe.value)
            pca_activations = state["activations"].select(pca_sample_ids) if pca_sample_ids else state["activations"]
            # Compute and cache PCA
            cache_key = (
                w_reader_method.value,
                tuple(sorted(pca_sample_ids)),
                w_pca_pool.value,
                w_pca_exclude_special.value,
            )
            if cache_key not in state["readers"]:
                state["readers"][cache_key] = compute_layerwise_score(
                    pca_activations,
                    method=w_reader_method.value,
                    pool_method=w_pca_pool.value,
                    exclude_bos=w_pca_exclude_bos.value,
                    exclude_special_tokens=w_pca_exclude_special.value,
                )

            # Select samples for visualization
            vis_sample_ids = list(w_vis_safe.value) + list(w_vis_unsafe.value)
            vis_activations = state["activations"].select(vis_sample_ids) if vis_sample_ids else state["activations"]
            # Plot activations
            plot_layerwise_score(
                vis_activations,
                readers=state["readers"][cache_key],
                pool_method=w_vis_pool.value,
                exclude_bos=w_vis_exclude_bos.value,
                exclude_special_tokens=w_vis_exclude_special.value,
            )

    for w in [
        w_reader_method,
        w_pca_safe,
        w_pca_unsafe,
        w_pca_pool,
        w_pca_exclude_bos,
        w_pca_exclude_special,
        w_vis_safe,
        w_vis_unsafe,
        w_vis_pool,
        w_vis_exclude_bos,
        w_vis_exclude_special,
    ]:
        w.observe(update_plot, names="value")

    # Display widgets
    display(widgets.VBox([layer1, widgets.HTML("<hr>"), layer2]))


analyze_layerwise_score()